# Mechanism Variants: Data Generation

The five dataset analyses each use one `mdatagen` method per mechanism. This notebook builds every other method the library exposes, so that `analysis_variants.ipynb` can ask a question none of those five can: does `missingfcup` distinguish mechanism *implementations*, or only mechanism *classes*?

Run this once to reproduce the CSV files in `data/`. It downloads the source datasets, so it needs network access.

## Which methods

All 21 that `mdatagen` 0.2.0 exposes.

| Class | Methods |
|---|---|
| `mMCAR` | `random`, `binomial` |
| `mMAR` | `random`, `correlated`, `median`, `pattern_missingness` |
| `mMNAR` | `random`, `correlated`, `median`, `MBOV_randomness`, `MBOV_median`, `MBIR`, `MBOUV` |
| `uMCAR` | `random`, `binomial` |
| `uMAR` | `highest`, `lowest`, `median`, `mix`, `rank` |
| `uMNAR` | `run` |

## Which datasets, and why only two

`titanic` and `breast_cancer`.

Most of this API selects rows by where a value sits in its column's order: `MBOV_median`, `MBOV_randomness`, `MBIR`, `mMNAR.median`, `mMAR.median` and all five `uMAR` methods. That is only meaningful on columns whose values are genuinely ordered.

| Dataset | Shape | Binary | Low-cardinality | Continuous |
|---|---|---|---|---|
| `titanic` | 712 x 5 | 0 | 2 | 3 |
| `breast_cancer` | 569 x 30 | 0 | 0 | 30 |
| `contraceptive_method` | 1473 x 9 | 3 | 4 | 2 |
| `student_performance` | 649 x 30 | 13 | 15 | 2 |

The two categorical datasets are excluded. Their columns are `pandas.factorize` output, where the integer codes are assigned by order of first appearance and carry no meaning. Asking such a method to remove the *lowest* values of a binary column means removing whichever category happened to appear second in the file: the call succeeds and the result is an artefact of row ordering. `student_performance` has 28 of its 30 columns in that state.

That leaves a narrow, mostly-continuous dataset and a wide, fully-continuous one, which is the contrast worth drawing, with every method semantically valid on both. `default_credit` is also out: at 30000 rows these variants would be about 60 MB of CSV against 2 MB for these two.

In [1]:
import os

import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.datasets import load_breast_cancer

from mdatagen.multivariate.mMCAR import mMCAR
from mdatagen.multivariate.mMAR import mMAR
from mdatagen.multivariate.mMNAR import mMNAR
from mdatagen.univariate.uMCAR import uMCAR
from mdatagen.univariate.uMAR import uMAR
from mdatagen.univariate.uMNAR import uMNAR

from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

MISSING_RATE = 20

## Loading

Each dataset is prepared exactly as its own analysis prepares it, with one addition: the index is reset to `0..n-1`. `uMAR.rank()` indexes positionally and raises `KeyError` on the gapped index that `dropna()` leaves behind. None of the five dataset notebooks call that method, so none of them hit it.

In [2]:
def prepare():
    """Return {name: (X, y, value_pair, x_miss, x_obs)}."""
    raw = sns.load_dataset("titanic").drop(columns=["deck"]).dropna()
    raw = raw.reset_index(drop=True)
    numeric = raw.select_dtypes(include=[np.number])

    bc = load_breast_cancer(as_frame=True)

    return {
        "titanic": (
            numeric.drop(columns=["survived"]),
            raw["survived"].to_numpy(),
            ["age", "fare"], "age", "pclass",
        ),
        "breast_cancer": (
            bc.data.reset_index(drop=True),
            bc.target.to_numpy(),
            ["mean radius", "mean area"], "mean radius", "mean texture",
        ),
    }


datasets = prepare()
{name: X.shape for name, (X, *_) in datasets.items()}

{'titanic': (712, 5), 'breast_cancer': (569, 30)}

## The variant table

Each entry is a callable, not a result, so nothing runs until the loop below invokes it under a fresh seed. `n_xmiss` follows what the dataset's own notebook uses: 3 affected columns for `titanic`, 15 for `breast_cancer`. The methods that take an explicit `columns=` list get the dataset's two named value columns.

In [3]:
def variants(X, y, pair, x_miss, x_obs):
    """Every mdatagen method, bound to one dataset."""
    n_xmiss = 3 if X.shape[1] <= 10 else 15
    rate = MISSING_RATE

    table = {
        "mcar_random": lambda: mMCAR(X=X, y=y, missing_rate=rate, seed=42).random(),
        "mcar_binomial": lambda: mMCAR(
            X=X, y=y, missing_rate=rate, seed=42).binomial(columns=pair),

        "mar_random": lambda: mMAR(X=X, y=y, n_xmiss=n_xmiss).random(missing_rate=rate),
        "mar_correlated": lambda: mMAR(
            X=X, y=y, n_xmiss=n_xmiss).correlated(missing_rate=rate),
        "mar_median": lambda: mMAR(X=X, y=y, n_xmiss=n_xmiss).median(missing_rate=rate),
        "mar_pattern": lambda: mMAR(X=X, y=y, n_xmiss=n_xmiss).pattern_missingness(
            missing_rate=rate, seed=42),

        "mnar_random": lambda: mMNAR(
            X=X, y=y, n_xmiss=n_xmiss, threshold=0).random(missing_rate=rate),
        "mnar_correlated": lambda: mMNAR(
            X=X, y=y, n_xmiss=n_xmiss, threshold=0).correlated(missing_rate=rate),
        "mnar_median": lambda: mMNAR(
            X=X, y=y, n_xmiss=n_xmiss, threshold=0).median(missing_rate=rate),
        "mnar_mbov_randomness": lambda: mMNAR(
            X=X, y=y, n_xmiss=2, threshold=0).MBOV_randomness(
                missing_rate=rate, randomness=0, columns=pair),
        "mnar_mbov_median": lambda: mMNAR(
            X=X, y=y, n_xmiss=2, threshold=0).MBOV_median(
                missing_rate=rate, columns=pair),
        "mnar_mbir": lambda: mMNAR(
            X=X, y=y, n_xmiss=2, threshold=0).MBIR(missing_rate=rate, columns=pair),
        "mnar_mbouv": lambda: mMNAR(
            X=X, y=y, n_xmiss=2, threshold=0).MBOUV(missing_rate=rate),

        "uni_mcar_random": lambda: uMCAR(
            X=X, y=y, missing_rate=rate, x_miss=x_miss, seed=42).random(),
        "uni_mcar_binomial": lambda: uMCAR(
            X=X, y=y, missing_rate=rate, x_miss=x_miss, seed=42).binomial(),
        "uni_mnar_run": lambda: uMNAR(
            X=X, y=y, missing_rate=rate, x_miss=x_miss).run(),
    }
    for method in ("highest", "lowest", "median", "mix", "rank"):
        table[f"uni_mar_{method}"] = (
            lambda method=method: getattr(
                uMAR(X=X, y=y, missing_rate=rate, x_miss=x_miss, x_obs=x_obs),
                method)()
        )
    return table


len(variants(*datasets["titanic"]))

21

## Generating

`np.random.seed(42)` is set immediately before every generator, the same rule the five dataset notebooks follow, so each variant is independent of what the others drew and the notebook reproduces its own output.

`mdatagen` appends a `target` column to some outputs and not others; it is dropped here so every file holds the same feature columns.

`mar_pattern` prints a warning about a missing `shift_lookup.csv` from inside `mdatagen`. It still returns a result, and the warning is not something this notebook can fix.

In [4]:
rows = []

for dataset, spec in datasets.items():
    os.makedirs(f"data/{dataset}", exist_ok=True)
    for name, build in variants(*spec).items():
        np.random.seed(42)
        frame = build()
        if isinstance(frame, tuple):
            frame = frame[0]
        frame = frame.drop(columns=["target"], errors="ignore")
        frame.to_csv(f"data/{dataset}/{name}.csv", index=False)

        affected = [c for c in frame.columns if frame[c].isna().any()]
        rows.append({
            "dataset": dataset,
            "variant": name,
            "columns_affected": len(affected),
            "dataset_rate": round(frame.isna().mean().mean(), 4),
            "rate_within_affected": round(frame[affected].isna().mean().mean(), 4)
            if affected else 0.0,
        })

summary = pd.DataFrame(rows)
print(f"wrote {len(summary)} files")

/Users/matias/.pyenv/versions/3.10.19/lib/python3.10/site-packages/mdatagen/multivariate/mMCAR.py:112: UserWarning: Binomial sometimes does not generate the input missing rate in dataset
  warnings.warn(


2026-08-24 19:52:46,579 [WARNING] Failed to load lookup table for a prespecified score to probability function. It is possible data/shift_lookup.csv is missing, in the wrong location, or corrupted. Try rerunning scripts/generate_shift_lookup_table.py to regenerate the lookup table.


/Users/matias/.pyenv/versions/3.10.19/lib/python3.10/site-packages/mdatagen/multivariate/mMCAR.py:112: UserWarning: Binomial sometimes does not generate the input missing rate in dataset
  warnings.warn(


2026-08-24 19:53:04,162 [WARNING] Failed to load lookup table for a prespecified score to probability function. It is possible data/shift_lookup.csv is missing, in the wrong location, or corrupted. Try rerunning scripts/generate_shift_lookup_table.py to regenerate the lookup table.


wrote 42 files


## Requested rate against delivered rate

Every variant asked for 20%. They do not all deliver it, and they do not all mean the same thing by it.

Methods that take an explicit `columns=` pair spread their 20% over those two columns only, so the dataset-wide rate falls as the dataset widens: the same call reads as 8% on five-column `titanic` and 1% on thirty-column `breast_cancer`. Reading `dataset_rate` alone would make those variants look inert on the wider dataset, which is why the rate within the affected columns is recorded beside it.

In [5]:
summary.pivot(index="variant", columns="dataset", values="dataset_rate")

dataset,breast_cancer,titanic
variant,,
mar_correlated,0.2004,0.2003
mar_median,0.2004,0.2003
mar_pattern,0.1134,0.0831
mar_random,0.2004,0.1997
mcar_binomial,0.0138,0.0826
mcar_random,0.2000,0.2000
mnar_correlated,0.2004,0.2003
mnar_mbir,0.0143,0.2999
mnar_mbouv,0.2000,0.2000


In [6]:
summary.pivot(index="variant", columns="dataset", values="rate_within_affected")

dataset,breast_cancer,titanic
variant,,
mar_correlated,0.4007,0.3338
mar_median,0.4007,0.3338
mar_pattern,0.2267,0.2079
mar_random,0.4007,0.3329
mcar_binomial,0.2074,0.2065
mcar_random,0.2000,0.2000
mnar_correlated,0.4007,0.3338
mnar_mbir,0.2004,0.5997
mnar_mbouv,0.2000,0.2000


In [7]:
summary.pivot(index="variant", columns="dataset", values="columns_affected")

dataset,breast_cancer,titanic
variant,,
mar_correlated,15,3
mar_median,15,3
mar_pattern,15,2
mar_random,15,3
mcar_binomial,2,2
mcar_random,30,5
mnar_correlated,15,3
mnar_mbir,2,2
mnar_mbouv,30,5


The files are now in `data/<dataset>/<variant>.csv`. `analysis_variants.ipynb` reads them and asks what the package can tell apart.